# AI Crawler Access Checker: robots.txt + llms.txt + bot rules

Companion notebook for the MLforSEO post **[Your robots.txt might be the silent killer of your AI traffic](https://www.mlforseo.com/machine-learning-implementation-guides/ai-search-optimisation/robots-txt-silent-killer-ai-traffic/)**. Built by Lazarina Stoy, MLforSEO.

A lot of sites added "block AI" rules to robots.txt in 2023 and 2024 and never looked at them again. Since then most AI companies have split their crawlers into **training bots** (collect data for models), **search bots** (index pages so an assistant can find and cite them) and **user-initiated fetchers** (visit a page because a person asked). Blocking the wrong one can quietly take you out of AI answers while doing nothing for the thing you actually wanted to stop.

For one or many domains, this notebook:

1. Fetches **robots.txt** and works out allow / block for 27 AI crawlers and control tokens, on the sample URLs you choose.
2. Checks **/llms.txt** and **/llms-full.txt**: are they there, and do they follow the basic [llmstxt.org](https://llmstxt.org/) format?
3. **Flags common mistakes**: blanket blocks, search bots blocked alongside training bots, Google-Extended misunderstandings, typos, outdated tokens.
4. Optionally requests a page **with each crawler's user-agent string** and records the status code.
5. **Exports** CSVs and one Sheets-ready Excel workbook.

**What you need:** nothing. No API keys, no accounts. Sample mode runs offline on saved robots.txt files; live mode needs internet access.

## 0. Setup

In Colab this installs [Protego](https://github.com/scrapy/protego) and downloads the helper module and sample files from the GitHub repo. Locally, run `pip install -r requirements.txt` from this folder first.

**Why Protego?** Python's built-in `urllib.robotparser` applies rules in file order and doesn't support the `*` and `$` wildcards the way Google does. Protego follows Google's robots.txt spec: the most specific (longest) matching rule wins, `Allow` wins a tie, and wildcards work. On top of that, the helper matches user-agent groups by the **exact product token**, as [RFC 9309](https://www.rfc-editor.org/rfc/rfc9309) and Google's own parser do.

In [1]:
import os, sys, subprocess, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "protego>=0.4"], check=True)

# Download the helper and the sample files if they aren't next to the notebook (e.g. in Colab)
RAW = "https://raw.githubusercontent.com/lazarinastoy/mlforseo-code/main/ai-crawler-access-checker/"
FILES = ["ai_crawler_access.py", "sample_data/domains.csv"] + [
    f"sample_data/robots/{d}.txt" for d in
    ["www.mlforseo.com", "www.nytimes.com", "www.theguardian.com", "www.bbc.co.uk",
     "www.reddit.com", "vercel.com", "example-shop.test"]
] + ["sample_data/llms/vercel.com__llms.txt", "sample_data/llms/example-shop.test__llms.txt",
     "sample_data/llms/www.reddit.com__llms.txt", "sample_data/llms/www.reddit.com__llms-full.txt"]
for f in FILES:
    if not Path(f).exists():
        Path(f).parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(RAW + f, f)

import pandas as pd
import ai_crawler_access as aca
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
print("Ready. Crawlers in the list:", len(aca.AI_AGENTS))

Ready. Crawlers in the list: 27


## 1. Settings

* `USE_SAMPLE = True` reads the saved robots.txt and llms.txt files in `sample_data/` (saved on 24 October 2026; these sites may have changed since). One of them, `example-shop.test`, is a fictional file packed with the mistakes this notebook looks for.
* `USE_SAMPLE = False` fetches everything live. Put your own domains in `DOMAINS` (with `www.` if your site uses it; robots.txt is per host).
* `PATHS` are the URLs tested for every crawler. Use a few that matter: the homepage, a blog post, a product page, a folder you deliberately block.

In [2]:
USE_SAMPLE = True

DOMAINS = [
    "www.mlforseo.com",
    "www.nytimes.com",
    "www.theguardian.com",
    "www.bbc.co.uk",
    "www.reddit.com",
    "vercel.com",
    "example-shop.test",   # fictional, sample mode only
]
PATHS = ["/", "/blog/", "/checkout/"]

RUN_LIVE_UA_CHECK = False                 # optional, see section 6
LIVE_UA_URL = "https://www.mlforseo.com/"

## 2. The crawler list (checked against each vendor's docs)

Every token below was checked against the vendor's own documentation on 24 October 2026. The `docs` column links to the source. Vendors add and rename crawlers every few months, so open the links before you change a live robots.txt.

| Vendor | Training | Search / answers | User-initiated | Source |
|---|---|---|---|---|
| OpenAI | GPTBot | OAI-SearchBot | ChatGPT-User | [OpenAI crawlers](https://developers.openai.com/api/docs/bots) |
| Anthropic | ClaudeBot | Claude-SearchBot | Claude-User | [Anthropic help centre](https://support.claude.com/en/articles/8896518-does-anthropic-crawl-data-from-the-web-and-how-can-site-owners-block-the-crawler) |
| Perplexity | (none) | PerplexityBot | Perplexity-User | [Perplexity crawlers](https://docs.perplexity.ai/guides/bots) |
| Google | Google-Extended (control token) | Googlebot (Search, incl. AI Overviews and AI Mode) | | [Google common crawlers](https://developers.google.com/search/docs/crawling-indexing/google-common-crawlers) |
| Apple | Applebot-Extended (control token) | Applebot | | [About Applebot](https://support.apple.com/en-us/119829) |
| Microsoft | | Bingbot | | [Bing crawlers](https://www.bing.com/webmasters/help/which-crawlers-does-bing-use-8c184ec0) |
| Amazon | Amazonbot | Amzn-SearchBot | Amzn-User | [Amazonbot](https://developer.amazon.com/amazonbot) |
| Meta | Meta-ExternalAgent | Meta-WebIndexer | Meta-ExternalFetcher | [Meta web crawlers](https://developers.facebook.com/docs/sharing/webmasters/web-crawlers/) |
| Mistral AI | MistralAI-Training | MistralAI-Index | MistralAI-User | [Mistral robots](https://docs.mistral.ai/robots) |
| DuckDuckGo | | DuckAssistBot | | [DuckAssistBot](https://duckduckgo.com/duckduckgo-help-pages/results/duckassistbot) |
| Common Crawl | CCBot | | | [CCBot](https://commoncrawl.org/ccbot) |
| ByteDance | Bytespider | | | no official page found |

Three things people get wrong most often:

* **Google-Extended does not control Google Search.** Google's docs say "Google-Extended does not impact a site's inclusion in Google Search nor is it used as a ranking signal in Google Search." It controls whether Google may use your content to train future Gemini models and for grounding in Gemini Apps and Vertex AI. Blocking it does **not** keep you out of AI Overviews or AI Mode; those run on Googlebot, and Google points to `nosnippet`, `data-nosnippet`, `max-snippet` or `noindex` to limit what they show.
* **Applebot-Extended and Google-Extended never crawl.** They are tokens a crawler reads in robots.txt to decide how the data can be used. Applebot-Extended opts you out of Apple's model training while keeping you in Siri and Spotlight; blocking **Applebot** removes you from those.
* **User-initiated fetchers may ignore robots.txt.** OpenAI says robots.txt rules "may not apply" to ChatGPT-User, and Perplexity says Perplexity-User "generally ignores" them. Amazon and Meta say similar things about their user fetchers. robots.txt is a request, not a lock.

In [3]:
aca.agents_table()

,vendor,token,category,controls,honours_robots,ua_source,docs
0,OpenAI,GPTBot,AI training,Crawls content that may be used to train OpenAI's generative AI foundation models.,yes,vendor docs,https://developers.openai.com/api/docs/bots
1,OpenAI,OAI-SearchBot,AI search / answers,Surfaces websites in ChatGPT's search features. Block it and your pages can't appear as ChatGPT search results.,yes,vendor docs,https://developers.openai.com/api/docs/bots
2,OpenAI,ChatGPT-User,User-initiated fetch,Visits a page when a ChatGPT or Custom GPT user asks it to.,may not apply (user-initiated),vendor docs,https://developers.openai.com/api/docs/bots
3,Anthropic,ClaudeBot,AI training,Collects public web content that may be used to train Anthropic's models.,yes,approximation,https://support.claude.com/en/articles/8896518-does-anthropic-crawl-data-from-the-web-and-how-can-site-owners-block-...
4,Anthropic,Claude-SearchBot,AI search / answers,Indexes pages to improve the quality of Claude's search results.,yes,approximation,https://support.claude.com/en/articles/8896518-does-anthropic-crawl-data-from-the-web-and-how-can-site-owners-block-...
5,Anthropic,Claude-User,User-initiated fetch,Fetches a page when a Claude user asks a question that needs it.,yes,approximation,https://support.claude.com/en/articles/8896518-does-anthropic-crawl-data-from-the-web-and-how-can-site-owners-block-...
6,Perplexity,PerplexityBot,AI search / answers,Surfaces and links websites in Perplexity search results. Perplexity says it is not used to crawl content for AI fou...,yes,vendor docs,https://docs.perplexity.ai/guides/bots
7,Perplexity,Perplexity-User,User-initiated fetch,Fetches a page when a Perplexity user asks a question.,generally ignores robots.txt (vendor docs),vendor docs,https://docs.perplexity.ai/guides/bots
8,Google,Googlebot,Search engine,"Google Search, including AI Overviews and AI Mode. Blocking it removes you from Search, AI features included.",yes,vendor docs,https://developers.google.com/search/docs/crawling-indexing/googlebot
9,Google,Google-Extended,Control token,Whether content Google crawls may be used to train future Gemini models and for grounding in Gemini Apps and Vertex ...,"yes (token only, no separate crawler)",NaN,https://developers.google.com/search/docs/crawling-indexing/google-common-crawlers


## 3. Run the audit

For each domain this fetches (or reads) robots.txt, `llms.txt` and `llms-full.txt`, evaluates every crawler on every path and runs the mistake checks. Live mode waits one second between domains to be polite.

How robots.txt status codes are treated (Google's rules): a **4xx** means "no robots.txt, crawl everything"; a **5xx or 429** means "stop crawling" until the file comes back.

In [4]:
result = aca.audit_domains(
    DOMAINS,
    paths=PATHS,
    offline_dir="sample_data" if USE_SAMPLE else None,
)
result.summary

,site,robots_status,robots_bytes,agents_blocked_on_home,agents_checked,training_bots,search_answer_bots,user_fetchers,googlebot,google_extended,llms_txt,llms_full_txt,high_flags,all_flags
0,https://www.mlforseo.com,200,120,0,27,0/7 blocked,0/7 blocked,0/6 blocked,allowed,allowed,False,False,0,0
1,https://www.nytimes.com,200,8547,17,27,5/7 blocked,5/7 blocked,4/6 blocked,allowed,blocked,False,False,0,10
2,https://www.theguardian.com,200,2873,12,27,5/7 blocked,4/7 blocked,1/6 blocked,allowed,allowed,False,False,0,6
3,https://www.bbc.co.uk,200,4907,15,27,6/7 blocked,3/7 blocked,3/6 blocked,allowed,blocked,False,False,0,7
4,https://www.reddit.com,200,538,27,27,7/7 blocked,7/7 blocked,6/6 blocked,blocked,blocked,False,False,2,10
5,https://vercel.com,200,575,0,27,0/7 blocked,0/7 blocked,0/6 blocked,allowed,allowed,True,False,0,0
6,https://example-shop.test,200,810,6,27,2/7 blocked,2/7 blocked,0/6 blocked,allowed,blocked,True,False,0,7


**How to read the summary:** `training_bots`, `search_answer_bots` and `user_fetchers` count how many crawlers of each type are blocked on the first path (usually the homepage). A site that blocks training bots but leaves search bots open has made a deliberate, consistent choice. A site where `search_answer_bots` are blocked too is invisible to most AI answer engines, whether it meant to be or not.

### Access matrix (first path)

In [5]:
matrix = aca.access_matrix(result.decisions)

def colour(v):
    return "background-color:#fde2e1" if v == "blocked" else ("background-color:#e3f4e8" if v == "allowed" else "")

# Transposed so crawlers are rows; easier to read with many crawlers
matrix.T.style.map(colour)

site,https://example-shop.test,https://vercel.com,https://www.bbc.co.uk,https://www.mlforseo.com,https://www.nytimes.com,https://www.reddit.com,https://www.theguardian.com
token,,,,,,,
GPTBot,blocked,allowed,blocked,allowed,blocked,blocked,allowed
OAI-SearchBot,blocked,allowed,blocked,allowed,blocked,blocked,allowed
ChatGPT-User,allowed,allowed,blocked,allowed,blocked,blocked,allowed
ClaudeBot,blocked,allowed,blocked,allowed,blocked,blocked,blocked
Claude-SearchBot,blocked,allowed,allowed,allowed,blocked,blocked,blocked
Claude-User,allowed,allowed,allowed,allowed,blocked,blocked,blocked
PerplexityBot,allowed,allowed,blocked,allowed,blocked,blocked,blocked
Perplexity-User,allowed,allowed,blocked,allowed,blocked,blocked,allowed
Googlebot,allowed,allowed,allowed,allowed,allowed,blocked,allowed


### Path-level decisions

Rules often differ by folder. This shows the decisions on every path you listed, only where at least one crawler is blocked. The `basis` column says which rule logic applied, and it tells you when a lenient, prefix-matching parser would reach a different answer.

In [6]:
d = result.decisions
by_path = d.pivot_table(index=["site", "token"], columns="path", values="decision", aggfunc="first")[PATHS]
by_path[(by_path == "blocked").any(axis=1)].head(40)

path                                                   /   /blog/ /checkout/
site                      token                                             
https://example-shop.test Amazonbot              allowed  allowed    blocked
                          Amzn-SearchBot         allowed  allowed    blocked
                          Amzn-User              allowed  allowed    blocked
                          Applebot               blocked  blocked    blocked
                          Applebot-Extended      allowed  allowed    blocked
                          Bingbot                allowed  allowed    blocked
                          Bytespider             allowed  allowed    blocked
                          CCBot                  allowed  allowed    blocked
                          ChatGPT-User           allowed  allowed    blocked
                          Claude-SearchBot       blocked  blocked    blocked
                          Claude-User            allowed  allowed    blocked
                          ClaudeBot              blocked  blocked    blocked
                          DuckAssistBot          allowed  allowed    blocked
                          GPTBot                 blocked  blocked    blocked
                          Google-CloudVertexBot  allowed  allowed    blocked
                          Google-Extended        blocked  blocked    blocked
                          Googlebot              allowed  allowed    blocked
                          Meta-ExternalAgent     allowed  allowed    blocked
                          Meta-ExternalFetcher   allowed  allowed    blocked
                          Meta-WebIndexer        allowed  allowed    blocked
                          MistralAI-Index        allowed  allowed    blocked
                          MistralAI-Training     allowed  allowed    blocked
                          MistralAI-User         allowed  allowed    blocked
                          OAI-SearchBot          blocked  blocked    blocked
                          Perplexity-User        allowed  allowed    blocked
                          PerplexityBot          allowed  allowed    blocked
                          facebookexternalhit    allowed  allowed    blocked
https://www.bbc.co.uk     Amazonbot              blocked  blocked    blocked
                          Amzn-SearchBot         blocked  blocked    blocked
                          Amzn-User              blocked  blocked    blocked
                          Applebot-Extended      blocked  blocked    blocked
                          Bytespider             blocked  blocked    blocked
                          CCBot                  blocked  blocked    blocked
                          ChatGPT-User           blocked  blocked    blocked
                          ClaudeBot              blocked  blocked    blocked
                          GPTBot                 blocked  blocked    blocked
                          Google-CloudVertexBot  blocked  blocked    blocked
                          Google-Extended        blocked  blocked    blocked
                          Meta-ExternalAgent     blocked  blocked    blocked
                          OAI-SearchBot          blocked  blocked    blocked

## 4. Flags: common mistakes

Severity guide: **high** = you are probably losing visibility you want; **medium** = worth a look, often a mismatch between intent and rule; **low / info** = housekeeping or a reminder of what a rule really does.

In [7]:
sev_order = {"high": 0, "medium": 1, "low": 2, "info": 3}
flags = result.flags.sort_values(["site", "severity"], key=lambda s: s.map(sev_order) if s.name == "severity" else s)
flags

,site,severity,issue,detail
34,https://example-shop.test,medium,"Applebot blocked, Applebot-Extended allowed","Blocking Applebot removes you from Siri, Spotlight and Safari search. To opt out of Apple's AI training only, allow ..."
35,https://example-shop.test,medium,OAI-SearchBot blocked along with GPTBot,"GPTBot is the training crawler; OAI-SearchBot is the search crawler. If you only meant to opt out of training, allow..."
36,https://example-shop.test,medium,Claude-SearchBot blocked along with ClaudeBot,"ClaudeBot is the training crawler; Claude-SearchBot is the search crawler. If you only meant to opt out of training,..."
38,https://example-shop.test,medium,Partial token 'Claude',"Matches the start of ClaudeBot, Claude-SearchBot, Claude-User. RFC 9309 and Google match the full token, so these cr..."
39,https://example-shop.test,medium,Possible typo: 'Perplexity-Bot',"Did you mean 'PerplexityBot'? Crawlers match the token exactly, so this group may be ignored."
37,https://example-shop.test,low,Outdated token 'anthropic-ai',"Not in Anthropic's current crawler docs. Use ClaudeBot (training), Claude-SearchBot and Claude-User."
33,https://example-shop.test,info,Google-Extended blocked (check the intent),This opts out of Gemini model training and grounding in Gemini Apps / Vertex AI. It does NOT remove you from Google ...
17,https://www.bbc.co.uk,medium,OAI-SearchBot blocked along with GPTBot,"GPTBot is the training crawler; OAI-SearchBot is the search crawler. If you only meant to opt out of training, allow..."
18,https://www.bbc.co.uk,medium,Amzn-SearchBot blocked along with Amazonbot,"Amazonbot is the training crawler; Amzn-SearchBot is the search crawler. If you only meant to opt out of training, a..."
19,https://www.bbc.co.uk,medium,PerplexityBot blocked,"PerplexityBot powers AI search answers that cite and link sources, not model training."


### The fictional example, explained

`example-shop.test` shows the typical "block AI" file written in a hurry:

* **GPTBot, OAI-SearchBot, ClaudeBot and Claude-SearchBot in one group.** That blocks training *and* search, so the shop can't be cited in ChatGPT search or Claude's search results.
* **Google-Extended blocked to "stay out of AI Overviews".** It doesn't do that. Only Googlebot (plus the snippet controls) affects AI Overviews.
* **Applebot blocked to stop Apple Intelligence training.** That removes the shop from Siri and Spotlight; `Applebot-Extended` was the token they wanted.
* **`Perplexity-Bot`**: a typo, so PerplexityBot ignores the group entirely.
* **`anthropic-ai` and `Claude`**: an outdated token and a partial one. Under exact token matching neither applies to ClaudeBot, Claude-SearchBot or Claude-User.

If what you want is "no training, yes to search and citations", the cleaner version looks like this:

```
User-agent: GPTBot
User-agent: ClaudeBot
User-agent: Google-Extended
User-agent: Applebot-Extended
User-agent: Meta-ExternalAgent
User-agent: CCBot
Disallow: /

User-agent: *
Disallow: /checkout/
Disallow: /cart/
```

OAI-SearchBot, Claude-SearchBot, PerplexityBot and the rest fall through to the `*` group and stay allowed. Decide for yourself which training bots belong on that list; this is a business decision as much as a technical one.

## 5. llms.txt and llms-full.txt

[llms.txt](https://llmstxt.org/) is a proposed Markdown file at the root of a site that gives language models a short guide to its most useful pages. The format: an H1 with the site name (the only required part), an optional `>` summary, then `##` sections with lists of links. `llms-full.txt` is a common companion that holds the full text.

Be realistic about it: it is a community proposal, not a standard, and the big AI search providers haven't documented that their crawlers read it. It is cheap to add and useful for agents and developer tools that look for it, but it does not override robots.txt and it won't fix a blocked crawler.

The check catches a common problem: a server that answers `/llms.txt` with **200 and an HTML page** (a soft 404), which looks like the file exists when it doesn't.

In [8]:
result.llms[["site", "file", "status", "present", "valid_basic", "h1_title", "has_summary",
             "h2_sections", "markdown_links", "size_bytes", "notes"]]

,site,file,status,present,valid_basic,h1_title,has_summary,h2_sections,markdown_links,size_bytes,notes
0,https://www.mlforseo.com,llms.txt,404,False,False,,False,0,0,NaN,not found (404)
1,https://www.mlforseo.com,llms-full.txt,404,False,False,,False,0,0,NaN,not found (404)
2,https://www.nytimes.com,llms.txt,404,False,False,,False,0,0,NaN,not found (404)
3,https://www.nytimes.com,llms-full.txt,404,False,False,,False,0,0,NaN,not found (404)
4,https://www.theguardian.com,llms.txt,404,False,False,,False,0,0,NaN,not found (404)
5,https://www.theguardian.com,llms-full.txt,404,False,False,,False,0,0,NaN,not found (404)
6,https://www.bbc.co.uk,llms.txt,404,False,False,,False,0,0,NaN,not found (404)
7,https://www.bbc.co.uk,llms-full.txt,404,False,False,,False,0,0,NaN,not found (404)
8,https://www.reddit.com,llms.txt,200,False,False,,False,0,0,216.0,"returns 200 but serves an HTML page or redirects to the homepage, so it is effectively missing (a 'soft 404')"
9,https://www.reddit.com,llms-full.txt,200,False,False,,False,0,0,216.0,"returns 200 but serves an HTML page or redirects to the homepage, so it is effectively missing (a 'soft 404')"


## 6. Optional: live user-agent check

This requests one URL with a normal browser user agent and then with each crawler's user-agent string (where one exists; control tokens have none), and records the status code and response size. A crawler that gets a 403 while the browser gets 200 points to a **user-agent rule in your CDN, firewall or server**, which robots.txt can't show you.

Read the results with care:

* **This only detects user-agent based blocking.** Cloudflare, Akamai, Fastly and most hosts verify real crawlers by IP range or reverse DNS. Your spoofed "GPTBot" request comes from your own IP, so it may be blocked as a fake bot even when the real GPTBot gets in, or let through when the real one is blocked. Check your CDN's bot settings for the real answer.
* If the **browser baseline** isn't 200, the site is blocking your network (common from Colab), and the per-bot results tell you little.
* Some user-agent strings are approximations because the vendor doesn't publish a full one (`ua_source` column).
* Only run this on sites you own or have permission to test. It makes one request per crawler with a one-second pause.

In [9]:
if RUN_LIVE_UA_CHECK:
    live = aca.live_ua_check(LIVE_UA_URL)
    display(live)
else:
    print("Skipped. Set RUN_LIVE_UA_CHECK = True and LIVE_UA_URL to one of your own pages to run it.")

Skipped. Set RUN_LIVE_UA_CHECK = True and LIVE_UA_URL to one of your own pages to run it.


## 7. Export

Writes one CSV per table and `ai_crawler_access_audit.xlsx` with a tab for each (summary, access matrix, decisions, flags, llms.txt, crawler list). Upload the `.xlsx` to Google Drive and open it with Google Sheets to share it with your team or a client.

In [10]:
written = aca.export(result, "results")
for p in written:
    print(p)

if IN_COLAB:
    from google.colab import files
    files.download("results/ai_crawler_access_audit.xlsx")

results/ai_crawler_summary.csv
results/ai_crawler_access_matrix.csv
results/ai_crawler_decisions.csv
results/ai_crawler_flags.csv
results/ai_crawler_llms_txt.csv
results/ai_crawler_crawler_list.csv
results/ai_crawler_access_audit.xlsx


## Limits and honest caveats

* **robots.txt is a request.** Well-behaved crawlers follow it; it doesn't stop anyone. Enforcement lives in your CDN or firewall.
* **The crawler list is a snapshot** (24 October 2026). New tokens appear and old ones get retired. The `LEGACY_TOKENS` and `AI_AGENTS` lists in `ai_crawler_access.py` are easy to edit.
* **Some sites serve different robots.txt files** to different user agents or IPs. This notebook sees the file served to it. For Google, Search Console's robots.txt report shows what Googlebot got.
* **Bytespider** has no official documentation we could find, so its robots.txt behaviour is unverified.
* **llms.txt checks are structural only.** They don't judge whether the links are the right ones.
* **Allowed in robots.txt doesn't mean cited.** Access is the first gate; whether an AI system uses your page depends on content, entities and authority.

## What to do next

* Read the paired post, **[Your robots.txt might be the silent killer of your AI traffic](https://www.mlforseo.com/machine-learning-implementation-guides/ai-search-optimisation/robots-txt-silent-killer-ai-traffic/)** by Beatrice Gamba, for the strategy behind these rules.
* Fix the high flags first, then decide training vs search per vendor and write it down, so the next person who edits robots.txt knows why each line is there.
* Go further with the course **[AI Search Optimization & Agentic SEO](https://www.mlforseo.com/courses/ai-search-optimisation-agentic-seo/)**, which covers crawler access as part of the wider agent-readiness stack.

Glossary: [robots.txt](https://www.mlforseo.com/glossary-term/robots-txt/) · [llms.txt](https://www.mlforseo.com/glossary-term/llms-txt/) · [AI Overview](https://www.mlforseo.com/glossary-term/ai-overview/) · [grounding](https://www.mlforseo.com/glossary-term/grounding/) · [training data](https://www.mlforseo.com/glossary-term/training-data/)